In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
cephasax_obdii_ds3_path = kagglehub.dataset_download('cephasax/obdii-ds3')

print('Data source import complete.')


100%|██████████| 7.39M/7.39M [00:00<00:00, 116MB/s]

Extracting files...


Data source import complete.


In [2]:
cephasax_obdii_ds3_path

'/root/.cache/kagglehub/datasets/cephasax/obdii-ds3/versions/3'

In [3]:
import os
from pathlib import Path

def explore_dataset_structure(base_path):
    """
    Recursively explores the dataset directory, listing folders, files, and their sizes.
    """
    dataset_dir = Path(base_path)

    # Check if path exists
    if not dataset_dir.exists():
        print(f"Error: The path '{base_path}' does not exist. Please check the path.")
        return

    print(f"--- Scanning Dataset at: {base_path} ---\n")

    total_files = 0
    total_size_bytes = 0

    # Walk through the directory tree
    for root, dirs, files in os.walk(dataset_dir):
        # Calculate level for pretty printing (indentation)
        level = str(root).replace(str(dataset_dir), "").count(os.sep)
        indent = "    " * level
        sub_indent = "    " * (level + 1)

        folder_name = os.path.basename(root)
        if root == str(dataset_dir):
            folder_name = dataset_dir.name

        print(f"{indent}[DIR] {folder_name}/")

        for filename in files:
            file_path = Path(root) / filename
            try:
                # Get file size in MB
                size_bytes = file_path.stat().st_size
                size_mb = size_bytes / (1024 * 1024)

                total_size_bytes += size_bytes
                total_files += 1

                print(f"{sub_indent}|-- {filename}  [Size: {size_mb:.2f} MB]")
            except Exception as e:
                print(f"{sub_indent}|-- {filename}  [Error reading size]")

    # Print Summary
    print(f"\n--- Summary ---")
    print(f"Total Files Found: {total_files}")
    print(f"Total Dataset Size: {total_size_bytes / (1024 * 1024):.2f} MB")
    print("----------------")

# TODO: Update this path to your actual dataset location on your machine
dataset_path = "/root/.cache/kagglehub/datasets/cephasax/obdii-ds3/versions/3"
explore_dataset_structure(dataset_path)

--- Scanning Dataset at: /root/.cache/kagglehub/datasets/cephasax/obdii-ds3/versions/3 ---

[DIR] 3/
    |-- exp1_14drivers_14cars_dailyRoutes.xlsx  [Size: 6.98 MB]
    |-- exp2_19drivers_1car_1route.xlsx  [Size: 1.15 MB]
    |-- exp3_4drivers_1car_1route.csv  [Size: 0.27 MB]
    |-- exp1_14drivers_14cars_dailyRoutes.csv  [Size: 6.73 MB]
    |-- exp2_19drivers_1car_1route.csv  [Size: 1.45 MB]
    |-- exp3_4drivers_1car_1route.xlsx  [Size: 0.24 MB]

--- Summary ---
Total Files Found: 6
Total Dataset Size: 16.81 MB
----------------


In [4]:
import pandas as pd
import os

# Define the file path based on the previous exploration
base_path = '/root/.cache/kagglehub/datasets/cephasax/obdii-ds3/versions/3/'
file_name = 'exp1_14drivers_14cars_dailyRoutes.csv'
file_path = os.path.join(base_path, file_name)

# Load the dataset
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded: {file_name}")
    print(f"Shape: {df.shape} (Rows, Columns)")

    # Display basic info to check data types and nulls
    print("\n--- Dataset Info ---")
    df.info()

    # Display first few rows to verify content
    print("\n--- First 5 Rows ---")
    print(df.head())

    # Analyze the Target Column (TROUBLE_CODES)
    print("\n--- Trouble Codes Distribution (Raw) ---")
    # value_counts with dropna=False to see the Nulls (Normal state)
    print(df['TROUBLE_CODES'].value_counts(dropna=False))

except FileNotFoundError:
    print(f"Error: File not found at {file_path}")

Successfully loaded: exp1_14drivers_14cars_dailyRoutes.csv
Shape: (60439, 33) (Rows, Columns)

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60439 entries, 0 to 60438
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   TIMESTAMP                    47514 non-null  float64
 1   MARK                         47459 non-null  object 
 2   MODEL                        47459 non-null  object 
 3   CAR_YEAR                     47459 non-null  float64
 4   ENGINE_POWER                 47459 non-null  object 
 5   AUTOMATIC                    47459 non-null  object 
 6   VEHICLE_ID                   47514 non-null  object 
 7   BAROMETRIC_PRESSURE(KPA)     10212 non-null  float64
 8   ENGINE_COOLANT_TEMP          33964 non-null  float64
 9   FUEL_LEVEL                   2994 non-null   object 
 10  ENGINE_LOAD                  30972 non-null  object 
 11  AMBIENT_AIR_TEMP

/tmp/ipython-input-413954235.py:11: DtypeWarning: Columns (1,2,4,5,6,9,10,14,15,16,20,21,22,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [5]:
import numpy as np
import pandas as pd # Ensure pandas is imported if running this cell independently

# 1. Function to clean numerical columns (remove % and swap , with .)
def clean_currency_fmt(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        x = x.replace('%', '').replace(',', '.')
    try:
        return float(x)
    except ValueError:
        return np.nan

# Only fix columns that we actually need based on the report
cols_to_fix = ['ENGINE_RPM', 'SPEED', 'ENGINE_COOLANT_TEMP']

print("--- Step 1: Cleaning Number Formats ---")
for col in cols_to_fix:
    if col in df.columns:
        df[col] = df[col].apply(clean_currency_fmt)
        print(f"Fixed format for: {col}")

# 2. Function to clean Target Label
def create_target(code):
    if pd.isna(code):
        return "Normal"
    code = str(code).strip()
    if code == "nan": return "Normal"

    # Take first 5 chars (standard OBD code)
    clean_code = code[:5]

    # Based on your previous data inspection:
    if clean_code == 'P0133':
        return 'P0133' # Oxygen Sensor Circuit Slow Response
    elif clean_code == 'C0300':
        return 'C0300' # Random/Multiple Cylinder Misfire (Chassis code variance)
    else:
        return "Other_Fault" # Grouping rare codes

print("\n--- Step 2: Creating Target Column ---")
df['Target'] = df['TROUBLE_CODES'].apply(create_target)
print("Target Distribution:")
print(df['Target'].value_counts())

# 3. Feature Selection & Alignment with Project Report
print("\n--- Step 3: Feature Alignment (Strict Report Compliance) ---")

# A. Create Constant Oil Pressure Column
# Rationale: Setting it to a constant value ensures the model ignores it during learning (Zero Variance),
# but the input shape remains compatible with the System Design (4 inputs).
df['Oil_Pressure_PSI'] = 30.0
print("Created 'Oil_Pressure_PSI' with constant value: 30.0")

# B. Select ONLY features mentioned in the Report (Section 5.1 & Sequence Diagrams)
# Source: RPM, Temp, Oil Pressure , Speed [cite: 74, 1026]
selected_features = [
    'ENGINE_RPM',
    'SPEED',
    'ENGINE_COOLANT_TEMP',
    'Oil_Pressure_PSI',  # Included for compatibility, ignored mathematically
    'Target'             # The Output
]

df_clean = df[selected_features].copy()

# 4. Handle Remaining Missing Values
print("\n--- Step 4: Handling Nulls ---")
original_len = len(df_clean)
# Drop rows where REAL sensors (RPM, Speed, Temp) are missing
df_clean = df_clean.dropna()
print(f"Rows before dropping nulls: {original_len}")
print(f"Rows after dropping nulls: {len(df_clean)}")

# Final check
print("\n--- Final Clean Dataset Info ---")
df_clean.info()
print("\n--- Sample Row (What the model will see) ---")
print(df_clean.head(1))

--- Step 1: Cleaning Number Formats ---
Fixed format for: ENGINE_RPM
Fixed format for: SPEED
Fixed format for: ENGINE_COOLANT_TEMP

--- Step 2: Creating Target Column ---
Target Distribution:
Target
Normal         48514
P0133           6070
C0300           5673
Other_Fault      182
Name: count, dtype: int64

--- Step 3: Feature Alignment (Strict Report Compliance) ---
Created 'Oil_Pressure_PSI' with constant value: 30.0

--- Step 4: Handling Nulls ---
Rows before dropping nulls: 60439
Rows after dropping nulls: 32809

--- Final Clean Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 32809 entries, 0 to 47512
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ENGINE_RPM           32809 non-null  float64
 1   SPEED                32809 non-null  float64
 2   ENGINE_COOLANT_TEMP  32809 non-null  float64
 3   Oil_Pressure_PSI     32809 non-null  float64
 4   Target               32809 non-null

In [6]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix

# 1. Setup Environment & Artifacts
artifacts_dir = 'artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

# Select Features exactly as per Report
# Note: Oil_Pressure_PSI is constant (30.0) but included for architectural compliance
feature_cols = ['ENGINE_RPM', 'SPEED', 'ENGINE_COOLANT_TEMP', 'Oil_Pressure_PSI']
X = df_clean[feature_cols]
y = df_clean['Target']

# 2. Preprocessing (Encoding & Splitting)
# Encode Targets
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Save Encoder immediately (Common for both approaches)
joblib.dump(le, os.path.join(artifacts_dir, 'label_encoder.joblib'))
joblib.dump(feature_cols, os.path.join(artifacts_dir, 'feature_order.joblib'))

# Split Data (Stratified to maintain fault ratios in test set)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scaling (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save Scaler (Common for both approaches)
joblib.dump(scaler, os.path.join(artifacts_dir, 'scaler.joblib'))

print(f" Data Prep Completed ")
print(f"Training Set: {X_train.shape[0]} samples")
print(f"Test Set: {X_test.shape[0]} samples")
print(f"Classes: {le.classes_}")

# Experiment 1: Class Weighting Approach
print("\n [Experiment 1] Training with Class Weights ")

# Train Random Forest with 'balanced' weights
# This automatically adjusts weights inversely proportional to class frequencies
rf_weighted = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_weighted.fit(X_train_scaled, y_train)

# Evaluate
y_pred_weighted = rf_weighted.predict(X_test_scaled)
f1_weighted = f1_score(y_test, y_pred_weighted, average='macro')
print(f"Experiment 1 (Weighted) Macro F1-Score: {f1_weighted:.4f}")


# Experiment 2: Targeted Under-sampling Approach
print("\n [Experiment 2] Training with Targeted Under-sampling ")

# Manual Under-sampling Logic:
# We downsample 'Normal' class ONLY to match the count of the largest fault class (e.g., P0133)
# We do NOT downsample everything to the smallest class (Other_Fault) to avoid massive data loss.

# Create a temporary dataframe for resampling
train_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
train_df['target'] = y_train

# Get counts
class_counts = train_df['target'].value_counts()
normal_class_idx = le.transform(['Normal'])[0]
normal_count = class_counts[normal_class_idx]

# Find largest minority class count (The target count for Normal)
minority_counts = class_counts.drop(normal_class_idx)
target_count = minority_counts.max()

print(f"Original Normal Count: {normal_count}")
print(f"Target Downsampling Count (Matches largest fault): {target_count}")

# Perform Sampling
sampled_dfs = []
for class_idx in class_counts.index:
    if class_idx == normal_class_idx:
        # Downsample Normal
        subset = train_df[train_df['target'] == class_idx].sample(n=target_count, random_state=42)
    else:
        # Keep Faults as is
        subset = train_df[train_df['target'] == class_idx]
    sampled_dfs.append(subset)

balanced_train_df = pd.concat(sampled_dfs).sample(frac=1, random_state=42) # Shuffle

X_train_under = balanced_train_df.drop('target', axis=1)
y_train_under = balanced_train_df['target']

# Train Random Forest (No class_weights needed now, data is balanced-ish)
rf_under = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_under.fit(X_train_under, y_train_under)

# Evaluate on the SAME original Test Set (Important!)
y_pred_under = rf_under.predict(X_test_scaled)
f1_under = f1_score(y_test, y_pred_under, average='macro')
print(f"Experiment 2 (Under-sampling) Macro F1-Score: {f1_under:.4f}")


# Final Comparison & Saving
print("\n Final Results & Selection ")

print("Detailed Report [Weighted Model]:")
print(classification_report(y_test, y_pred_weighted, target_names=le.classes_))

print("Detailed Report [Under-sampled Model]:")
print(classification_report(y_test, y_pred_under, target_names=le.classes_))

# Select Winner
if f1_weighted >= f1_under:
    print(f" Winner: Class Weighting Strategy (F1: {f1_weighted:.4f})")
    best_model = rf_weighted
    method_name = "weighted"
else:
    print(f" Winner: Under-sampling Strategy (F1: {f1_under:.4f})")
    best_model = rf_under
    method_name = "undersampling"

# Save the Best Model
model_path = os.path.join(artifacts_dir, 'best_model_rf.joblib')
joblib.dump(best_model, model_path)
print(f" Best model saved to: {model_path}")

 Data Prep Completed 
Training Set: 26247 samples
Test Set: 6562 samples
Classes: ['C0300' 'Normal' 'Other_Fault' 'P0133']

 [Experiment 1] Training with Class Weights 
Experiment 1 (Weighted) Macro F1-Score: 0.5259

 [Experiment 2] Training with Targeted Under-sampling 
Original Normal Count: 16877
Target Downsampling Count (Matches largest fault): 4750


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Experiment 2 (Under-sampling) Macro F1-Score: 0.5249

 Final Results & Selection 
Detailed Report [Weighted Model]:
              precision    recall  f1-score   support

       C0300       0.63      0.60      0.62      1122
      Normal       0.88      0.90      0.89      4220
 Other_Fault       0.00      0.00      0.00        33
       P0133       0.60      0.59      0.59      1187

    accuracy                           0.79      6562
   macro avg       0.53      0.52      0.53      6562
weighted avg       0.78      0.79      0.79      6562

Detailed Report [Under-sampled Model]:
              precision    recall  f1-score   support

       C0300       0.55      0.67      0.60      1122
      Normal       0.93      0.82      0.87      4220
 Other_Fault       0.06      0.03      0.04        33
       P0133       0.53      0.66      0.59      1187

    accuracy                           0.76      6562
   macro avg       0.52      0.54      0.52      6562
weighted avg       0.79      0

In [7]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

# Setup
artifacts_dir = 'artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

# 1. Load & Basic Clean (Same as before)
# Assuming df is already loaded from previous cells
# Re-apply basic cleaning just to be safe
cols_to_fix = ['ENGINE_RPM', 'SPEED', 'ENGINE_COOLANT_TEMP']
for col in cols_to_fix:
    if col in df.columns:
        df[col] = df[col].apply(clean_currency_fmt)

# 2. Advanced Target Creation (Remove Weak Classes)
def create_target_strict(code):
    if pd.isna(code): return "Normal"
    code = str(code).strip()
    if code == "nan": return "Normal"
    clean_code = code[:5]

    # We ONLY keep the classes we have enough data for
    if clean_code in ['P0133', 'C0300']:
        return clean_code
    else:
        return "Drop" # Mark rare faults for deletion

df['Target'] = df['TROUBLE_CODES'].apply(create_target_strict)

# Filter out the 'Drop' rows (Rare faults confuse the model)
df_clean = df[df['Target'] != 'Drop'].copy()

# 3. Smart Feature Engineering (The Secret Sauce) 🌶️
# We derive these FROM the Report inputs. User input doesn't change.

# Feature A: Gear Ratio Proxy / Load Indicator
# Logic: High RPM at Low Speed = Low Gear or High Load
# Add small epsilon to avoid division by zero
df_clean['Ratio_RPM_Speed'] = df_clean['ENGINE_RPM'] / (df_clean['SPEED'] + 1)

# Feature B: Thermal Intensity
# Logic: High RPM + High Temp is dangerous
df_clean['Thermal_Stress'] = df_clean['ENGINE_RPM'] * df_clean['ENGINE_COOLANT_TEMP']

# Feature C: Idle Indicator
# Logic: Speed is 0 but RPM > 0
df_clean['Is_Idle'] = (df_clean['SPEED'] < 1).astype(int)

# Feature D: The Report Dummy (Oil Pressure) - Kept for compliance
df_clean['Oil_Pressure_PSI'] = 30.0

# Select Features: 3 from Report + 3 Engineered + 1 Dummy
feature_cols = [
    'ENGINE_RPM', 'SPEED', 'ENGINE_COOLANT_TEMP', # User Inputs
    'Oil_Pressure_PSI', # Dummy Input
    'Ratio_RPM_Speed', 'Thermal_Stress', 'Is_Idle' # Internal Features
]

# Drop Nulls
df_clean = df_clean.dropna(subset=feature_cols)

print(f"Dataset Size after cleaning: {len(df_clean)}")
print("Target Distribution:\n", df_clean['Target'].value_counts())

# 4. Prepare for Training
X = df_clean[feature_cols]
y = df_clean['Target']

# Encode
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Training (Using Class Weights is usually better given previous results)
print("\n--- Training Improved Model ---")
rf_model = RandomForestClassifier(
    n_estimators=200,          # More trees
    max_depth=20,              # Deeper trees to catch subtle patterns
    class_weight='balanced',   # Handle imbalance
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

# 6. Evaluation
y_pred = rf_model.predict(X_test_scaled)

print("\nDetailed Report [Improved Model]:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 7. Save Artifacts (Critical for Deployment)
# We need to save a custom function logic? No, we implement the logic in the API code.
# We just save the scaler, encoder, and model.
joblib.dump(le, os.path.join(artifacts_dir, 'label_encoder.joblib'))
joblib.dump(scaler, os.path.join(artifacts_dir, 'scaler.joblib'))
joblib.dump(rf_model, os.path.join(artifacts_dir, 'best_model_rf.joblib'))
# Save the LIST of columns expected by the scaler (Internal features included)
joblib.dump(feature_cols, os.path.join(artifacts_dir, 'internal_feature_list.joblib'))

print(" Model and Artifacts Saved.")

Dataset Size after cleaning: 32646
Target Distribution:
 Target
Normal    21097
P0133      5937
C0300      5612
Name: count, dtype: int64

--- Training Improved Model ---

Detailed Report [Improved Model]:
              precision    recall  f1-score   support

       C0300       0.62      0.64      0.63      1122
      Normal       0.90      0.90      0.90      4220
       P0133       0.62      0.61      0.62      1188

    accuracy                           0.80      6530
   macro avg       0.72      0.72      0.72      6530
weighted avg       0.80      0.80      0.80      6530

 Model and Artifacts Saved.


In [8]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Setup Artifacts for Binary Model
binary_artifacts_dir = 'artifacts_binary'
os.makedirs(binary_artifacts_dir, exist_ok=True)

# 1. Target Creation (Binary Mode)
def create_binary_target(code):
    if pd.isna(code): return "Normal"
    code = str(code).strip()
    if code == "nan": return "Normal"

    clean_code = code[:5]

    # Logic: Any defined fault is 'Fault', otherwise 'Normal'
    # We still filter strictly to avoid noise from rare undefined codes
    if clean_code in ['P0133', 'C0300']:
        return "Fault"
    elif clean_code == "Normal": # Explicit Normal
        return "Normal"
    else:
        return "Drop" # Still drop rare ambiguous codes

# Apply Binary Mapping
df['Binary_Target'] = df['TROUBLE_CODES'].apply(create_binary_target)

# Filter Drop
df_binary = df[df['Binary_Target'] != 'Drop'].copy()

# 2. Internal Feature Engineering (Keep the successful logic)
df_binary['Ratio_RPM_Speed'] = df_binary['ENGINE_RPM'] / (df_binary['SPEED'] + 1)
df_binary['Thermal_Stress'] = df_binary['ENGINE_RPM'] * df_binary['ENGINE_COOLANT_TEMP']
df_binary['Is_Idle'] = (df_binary['SPEED'] < 1).astype(int)
df_binary['Oil_Pressure_PSI'] = 30.0

# Features List
feature_cols = [
    'ENGINE_RPM', 'SPEED', 'ENGINE_COOLANT_TEMP',
    'Oil_Pressure_PSI',
    'Ratio_RPM_Speed', 'Thermal_Stress', 'Is_Idle'
]

# Drop Nulls
df_binary = df_binary.dropna(subset=feature_cols)

print(f"Binary Dataset Size: {len(df_binary)}")
print("Binary Distribution:\n", df_binary['Binary_Target'].value_counts())

# 3. Prep Data
X = df_binary[feature_cols]
y = df_binary['Binary_Target']

# Encode (Normal vs Fault)
le_binary = LabelEncoder()
y_encoded = le_binary.fit_transform(y)
# Note: Usually Normal=1, Fault=0 or vice versa depending on alphabetical order (Fault, Normal) -> Fault=0

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scale
scaler_binary = StandardScaler()
X_train_scaled = scaler_binary.fit_transform(X_train)
X_test_scaled = scaler_binary.transform(X_test)

# 4. Training Binary Model
print("\n--- Training Binary Random Forest ---")
rf_binary = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced', # Critical for detection recall
    random_state=42,
    n_jobs=-1
)

rf_binary.fit(X_train_scaled, y_train)

# 5. Evaluation
y_pred = rf_binary.predict(X_test_scaled)

print("\nDetailed Report [Binary Model]:")
print(classification_report(y_test, y_pred, target_names=le_binary.classes_))

# Confusion Matrix to see False Negatives (Safety Critical)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"\nConfusion Matrix Stats:")
print(f"Correctly Detected Faults (TP): {tp}")
print(f"Missed Faults (FN - DANGER): {fn}")
print(f"False Alarms (FP): {fp}")

# 6. Save Binary Artifacts
joblib.dump(le_binary, os.path.join(binary_artifacts_dir, 'label_encoder.joblib'))
joblib.dump(scaler_binary, os.path.join(binary_artifacts_dir, 'scaler.joblib'))
joblib.dump(rf_binary, os.path.join(binary_artifacts_dir, 'binary_model_rf.joblib'))
joblib.dump(feature_cols, os.path.join(binary_artifacts_dir, 'internal_feature_list.joblib'))

print("\n Binary Model Saved.")

Binary Dataset Size: 32646
Binary Distribution:
 Binary_Target
Normal    21097
Fault     11549
Name: count, dtype: int64

--- Training Binary Random Forest ---

Detailed Report [Binary Model]:
              precision    recall  f1-score   support

       Fault       0.83      0.83      0.83      2310
      Normal       0.91      0.91      0.91      4220

    accuracy                           0.88      6530
   macro avg       0.87      0.87      0.87      6530
weighted avg       0.88      0.88      0.88      6530


Confusion Matrix Stats:
Correctly Detected Faults (TP): 3839
Missed Faults (FN - DANGER): 381
False Alarms (FP): 402

 Binary Model Saved.


In [11]:
import os
import joblib
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    auc,
    precision_recall_curve
)

#  Configuration & Setup
sns.set_style("whitegrid") # Professional plot style
BINARY_SOURCE = 'artifacts_binary'
MULTICLASS_SOURCE = 'artifacts'

DEPLOYMENT_DIR = 'final_deployment_v1'
# Create structure
dirs_to_create = [
    DEPLOYMENT_DIR,
    os.path.join(DEPLOYMENT_DIR, 'models'),
    os.path.join(DEPLOYMENT_DIR, 'encoders'),
    os.path.join(DEPLOYMENT_DIR, 'reports'),
    os.path.join(DEPLOYMENT_DIR, 'plots') # New folder for images
]
for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)

print(f" Starting Comprehensive Evaluation & Archiving Process...\n")

#  Helper Functions for Visualization
def plot_conf_matrix(y_true, y_pred, classes, title, save_path):
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_feat_importance(model, feature_names, title, save_path):
    plt.figure(figsize=(10, 6))
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]
        sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices], palette='viridis')
        plt.title(title)
        plt.xlabel('Relative Importance')
        plt.tight_layout()
        plt.savefig(save_path)
    plt.close()

def display_model_params(model, model_name):
    print(f"\n🔹 {model_name} Hyperparameters:")
    params = model.get_params()
    # Filter for interesting params to keep output clean
    interesting_keys = ['n_estimators', 'max_depth', 'min_samples_split', 'class_weight', 'criterion']
    filtered_params = {k: params[k] for k in interesting_keys if k in params}
    df_params = pd.DataFrame.from_dict(filtered_params, orient='index', columns=['Value'])
    print(df_params)
    print("-" * 40)

# 1. EVALUATE BINARY MODEL (The Gatekeeper)
print(f"\n  PHASE 1: BINARY MODEL EVALUATION (Gatekeeper) ")

# Load Components
rf_binary = joblib.load(os.path.join(BINARY_SOURCE, 'binary_model_rf.joblib'))
scaler_binary = joblib.load(os.path.join(BINARY_SOURCE, 'scaler.joblib'))
le_binary = joblib.load(os.path.join(BINARY_SOURCE, 'label_encoder.joblib'))
feats_binary = joblib.load(os.path.join(BINARY_SOURCE, 'internal_feature_list.joblib'))

# Display Model Type & Params
print(f"Model Type: {type(rf_binary).__name__}")
display_model_params(rf_binary, "Binary Random Forest")

# Try to Evaluate
try:
    # NOTE: Assuming X_test_scaled and y_test exist in memory from the previous binary cell run.
    # If not, we skip dynamic plotting but still save artifacts.
    y_pred_bin = rf_binary.predict(X_test_scaled)
    y_probs_bin = rf_binary.predict_proba(X_test_scaled)[:, 1] # Probability for ROC
    y_true_bin = y_test

    # 1. Metrics
    print(">> Calculating Metrics...")
    acc_bin = accuracy_score(y_true_bin, y_pred_bin)
    f1_bin = f1_score(y_true_bin, y_pred_bin, average='weighted')
    report_bin = classification_report(y_true_bin, y_pred_bin, target_names=le_binary.classes_, output_dict=True)

    print(f" Accuracy: {acc_bin:.4f}")
    print(f" Weighted F1: {f1_bin:.4f}")

    # 2. Visualizations
    print(">> Generating Plots...")
    # Confusion Matrix
    plot_conf_matrix(y_true_bin, y_pred_bin, le_binary.classes_,
                     "Binary Model Confusion Matrix",
                     os.path.join(DEPLOYMENT_DIR, 'plots', 'binary_confusion_matrix.png'))

    # Feature Importance
    plot_feat_importance(rf_binary, feats_binary,
                         "Binary Model Feature Importance",
                         os.path.join(DEPLOYMENT_DIR, 'plots', 'binary_feature_importance.png'))

    # ROC Curve (Specific to Binary)
    fpr, tpr, _ = roc_curve(y_true_bin, y_probs_bin)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (Binary)')
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(DEPLOYMENT_DIR, 'plots', 'binary_roc_curve.png'))
    plt.close()

    # Save Report JSON
    with open(os.path.join(DEPLOYMENT_DIR, 'reports', 'binary_report.json'), 'w') as f:
        json.dump(report_bin, f, indent=4)

except NameError:
    print(" Warning: Binary Test Data variables not found in memory. Visuals skipped.")
except Exception as e:
    print(f" Warning during evaluation: {e}")

# Copy Artifacts
joblib.dump(rf_binary, os.path.join(DEPLOYMENT_DIR, 'models', 'gatekeeper_model.joblib'))
joblib.dump(scaler_binary, os.path.join(DEPLOYMENT_DIR, 'encoders', 'binary_scaler.joblib'))
joblib.dump(le_binary, os.path.join(DEPLOYMENT_DIR, 'encoders', 'binary_label_encoder.joblib'))
joblib.dump(feats_binary, os.path.join(DEPLOYMENT_DIR, 'encoders', 'binary_features.joblib'))


# 2. EVALUATE MULTI-CLASS MODEL (The Specialist)
print(f"\n  PHASE 2: MULTI-CLASS MODEL EVALUATION (Specialist) ")

# Load Components
rf_multi = joblib.load(os.path.join(MULTICLASS_SOURCE, 'best_model_rf.joblib'))
scaler_multi = joblib.load(os.path.join(MULTICLASS_SOURCE, 'scaler.joblib'))
le_multi = joblib.load(os.path.join(MULTICLASS_SOURCE, 'label_encoder.joblib'))
feats_multi = joblib.load(os.path.join(MULTICLASS_SOURCE, 'internal_feature_list.joblib'))

# Display Model Type & Params
print(f"Model Type: {type(rf_multi).__name__}")
display_model_params(rf_multi, "Multi-class Random Forest")

# Try to Load Test Data specifically for Multi-class
try:
    # Attempt to load saved test data if available from the previous step
    test_data_path = os.path.join(MULTICLASS_SOURCE, 'test_data.joblib')
    if os.path.exists(test_data_path):
        print(f">> Loading Multi-class Test Data from: {test_data_path}")
        X_test_multi, y_test_multi = joblib.load(test_data_path)

        # Predict
        y_pred_multi = rf_multi.predict(X_test_multi)

        # 1. Metrics
        print(">> Calculating Metrics...")
        acc_multi = accuracy_score(y_test_multi, y_pred_multi)
        f1_multi = f1_score(y_test_multi, y_pred_multi, average='weighted')
        report_multi = classification_report(y_test_multi, y_pred_multi, target_names=le_multi.classes_, output_dict=True)

        print(f" Accuracy: {acc_multi:.4f}")
        print(f" Weighted F1: {f1_multi:.4f}")

        # 2. Visualizations
        print(">> Generating Plots...")
        # Confusion Matrix
        plot_conf_matrix(y_test_multi, y_pred_multi, le_multi.classes_,
                         "Multi-class Confusion Matrix",
                         os.path.join(DEPLOYMENT_DIR, 'plots', 'multiclass_confusion_matrix.png'))

        # Feature Importance
        plot_feat_importance(rf_multi, feats_multi,
                             "Multi-class Feature Importance",
                             os.path.join(DEPLOYMENT_DIR, 'plots', 'multiclass_feature_importance.png'))

        # Save Report JSON
        with open(os.path.join(DEPLOYMENT_DIR, 'reports', 'multiclass_report.json'), 'w') as f:
            json.dump(report_multi, f, indent=4)

    else:
        print(" Multi-class test data file not found. Skipping dynamic plots.")

except Exception as e:
    print(f" Error evaluating multi-class model: {e}")

# Copy Artifacts
joblib.dump(rf_multi, os.path.join(DEPLOYMENT_DIR, 'models', 'specialist_model.joblib'))
joblib.dump(scaler_multi, os.path.join(DEPLOYMENT_DIR, 'encoders', 'multi_scaler.joblib'))
joblib.dump(le_multi, os.path.join(DEPLOYMENT_DIR, 'encoders', 'multi_label_encoder.joblib'))
joblib.dump(feats_multi, os.path.join(DEPLOYMENT_DIR, 'encoders', 'multi_features.joblib'))

print(" Multi-class Artifacts Archived.")


# 3. GENERATE ARCHITECTURE DOCUMENTATION (Meta-data)
print(f"\n  PHASE 3: METADATA GENERATION ")

system_meta = {
    "version": "1.0.0",
    "deployment_date": pd.Timestamp.now().isoformat(),
    "architecture": "Cascade (Binary Gatekeeper -> Multi-class Specialist)",
    "input_schema": {
        "user_inputs": ["ENGINE_RPM", "SPEED", "ENGINE_COOLANT_TEMP"],
        "system_inputs": ["Oil_Pressure_PSI (Constant=30)"],
        "derived_features": ["Ratio_RPM_Speed", "Thermal_Stress", "Is_Idle"]
    },
    "models": {
        "gatekeeper": {
            "file": "models/gatekeeper_model.joblib",
            "type": "RandomForestClassifier",
            "classes": list(le_binary.classes_),
            "metrics": {"f1_weighted": f1_bin if 'f1_bin' in locals() else "N/A"}
        },
        "specialist": {
            "file": "models/specialist_model.joblib",
            "type": "RandomForestClassifier",
            "classes": list(le_multi.classes_),
            "metrics": {"f1_weighted": f1_multi if 'f1_multi' in locals() else "N/A"}
        }
    }
}

with open(os.path.join(DEPLOYMENT_DIR, 'system_metadata.json'), 'w') as f:
    json.dump(system_meta, f, indent=4)

print(" System Metadata Saved.")


# 4. FINAL STRUCTURE SUMMARY
print(f"\n FINAL DEPLOYMENT PACKAGE READY AT: '{DEPLOYMENT_DIR}'")
print(f"   (Includes Models, Encoders, Reports, and Plots)")
for root, dirs, files in os.walk(DEPLOYMENT_DIR):
    level = root.replace(DEPLOYMENT_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

print(f"\n DONE. You can now download the '{DEPLOYMENT_DIR}' folder.")

 Starting Comprehensive Evaluation & Archiving Process...


  PHASE 1: BINARY MODEL EVALUATION (Gatekeeper) 
Model Type: RandomForestClassifier

🔹 Binary Random Forest Hyperparameters:
                      Value
n_estimators            200
max_depth                20
min_samples_split         2
class_weight       balanced
criterion              gini
----------------------------------------
>> Calculating Metrics...
 Accuracy: 0.8801
 Weighted F1: 0.8800
>> Generating Plots...


/tmp/ipython-input-3416828984.py:56: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices], palette='viridis')



  PHASE 2: MULTI-CLASS MODEL EVALUATION (Specialist) 
Model Type: RandomForestClassifier

🔹 Multi-class Random Forest Hyperparameters:
                      Value
n_estimators            200
max_depth                20
min_samples_split         2
class_weight       balanced
criterion              gini
----------------------------------------
 Multi-class test data file not found. Skipping dynamic plots.
 Multi-class Artifacts Archived.

  PHASE 3: METADATA GENERATION 
 System Metadata Saved.

 FINAL DEPLOYMENT PACKAGE READY AT: 'final_deployment_v1'
   (Includes Models, Encoders, Reports, and Plots)
final_deployment_v1/
    system_metadata.json
    reports/
        binary_report.json
    encoders/
        multi_features.joblib
        multi_label_encoder.joblib
        binary_scaler.joblib
        binary_features.joblib
        binary_label_encoder.joblib
        multi_scaler.joblib
    plots/
        binary_roc_curve.png
        binary_feature_importance.png
        binary_confusion_

In [12]:
import os

folder_to_zip = '/content/final_deployment_v1'
output_zip_file = '/content/final_deployment_v1.zip'

# Ensure the folder exists before attempting to zip
if os.path.exists(folder_to_zip):
    print(f"Zipping folder: {folder_to_zip} to {output_zip_file}")
    # Use shutil.make_archive for a more robust Pythonic way, or os.system with zip command
    # For simplicity and direct command execution similar to shell, using os.system
    os.system(f'zip -r {output_zip_file} {folder_to_zip}')
    print("Zipping complete.")
    print(f"You can now download the zip file from: {output_zip_file}")
else:
    print(f"Error: Folder '{folder_to_zip}' not found. Cannot zip.")

Zipping folder: /content/final_deployment_v1 to /content/final_deployment_v1.zip
Zipping complete.
You can now download the zip file from: /content/final_deployment_v1.zip
